# Semantle LoReFT Experiments

Interactive notebook to launch and compare LoReFT / Distributional intervention training runs on the Semantle word-search task.

**Sections**
1. Setup & imports  
2. Config — edit all hyperparameters here  
3. Load data  
4. Run training  
5. Inspect learned embeddings (post-training)


## 1. Setup

In [1]:
import os, sys, importlib

REPO_ROOT = "/work/pi_mccallum_umass_edu/jkarnuthala_umass_edu/D-Intervention"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import torch

import pyreft.reft_trainer
importlib.reload(pyreft.reft_trainer)

# Always reload so edits to semantle_reft_bo.py take effect without kernel restart - not working(to recheck)
import semantle_reft_bo
importlib.reload(semantle_reft_bo)
from semantle_reft_bo import (
    load_semantle_csv,
    train_semantle_reft,
)

print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

nnsight is not detected. Please install via 'pip install nnsight' for nnsight backend.
torch: 2.5.1 | CUDA: True
GPU: NVIDIA A16


## 2. Config — edit all hyperparameters here

In [1]:
# ── Data ──────────────────────────────────────────────────────────────────────
SEMANTLE_CSV = "/work/pi_mccallum_umass_edu/jkarnuthala_umass_edu/BOPRO-ICLR-2025/data/semantle/train/computer.csv"
TOP_K        = 8   # how many words to use from the CSV

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_NAME  = "meta-llama/Llama-3.2-1B"
CACHE_DIR   = "/datasets/ai/llama3/hub"  

# ── Intervention ──────────────────────────────────────────────────────────────
INTERVENTION_TYPE = "DistributionalWordIntervention"   # or "LoreftWordIntervention"
LAYER         = 10       # which transformer layer to intervene on
LOW_RANK_DIM  = 64       # rank r of the ReFT projection
POSITION      = "l1"     # token position: "l1", "f1", "f1+l1"

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS       = 700
BATCH_SIZE   = 4
LR           = 1e-3
SEED         = 42
# LR scheduler: "linear" (default, decays to 0), "cosine", "constant",
#               "cosine_with_restarts", "polynomial", "constant_with_warmup"
LR_SCHEDULER = "linear"
WARMUP_RATIO = 0.00   # fraction of total steps used for warmup (0 = no warmup)

# ── Distributional-only knobs ──────────────────────────────────────────────────
BETA       = 0.1    # KL divergence weight (β · KL). Set 0 to disable KL.

#not being used now, but pass the param to the model
LAMBDA_SEM = 0.0    # semantic structure loss weight. Set >0 to pull similar words together.

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = "./out_notebook"

# ── Weights & Biases ──────────────────────────────────────────────────────────
WANDB_PROJECT  = "semantle-reft"          # set to None to disable W&B
WANDB_RUN_NAME = "8_words-LR-1e-3-layer-10-rank64-beta0.1-linearlr" # human-readable run label

# ── Qualitative evaluation ────────────────────────────────────────────────────
# Every EVAL_STEPS training steps, print top-5 predicted tokens for a stratified
# sample of 8 words so you can eyeball whether the learned bias vectors are
# converging toward the right words. Set to 0 to disable.
EVAL_STEPS = 100

print("Config ready.")

Config ready.


## 3. Load data

In [3]:
target_word, words, sim_map = load_semantle_csv(SEMANTLE_CSV, top_k=TOP_K)
print(f"Target word : {target_word}")
print(f"Vocabulary  : {len(words)} words")


Target word : laptop
Vocabulary  : 8 words


## 4. Run training

In [4]:
# Edit any config cell above, then re-run this cell to kick off training.
saved_dir = train_semantle_reft(
    model_name        = MODEL_NAME,
    words             = words,
    intervention_type = INTERVENTION_TYPE,
    low_rank_dim      = LOW_RANK_DIM,
    layer             = LAYER,
    position          = POSITION,
    epochs            = EPOCHS,
    batch_size        = BATCH_SIZE,
    lr                = LR,
    lr_scheduler_type = LR_SCHEDULER,
    warmup_ratio      = WARMUP_RATIO,
    beta              = BETA,
    lambda_sem        = LAMBDA_SEM,
    output_dir        = OUTPUT_DIR,
    cache_dir         = CACHE_DIR,
    wandb_project     = WANDB_PROJECT,
    wandb_run_name    = WANDB_RUN_NAME,
    seed              = SEED,
    eval_steps        = EVAL_STEPS,
)
print("Training done. Saved to:", saved_dir)

trainable intervention params: 263,232 || trainable model params: 0
model params: 1,235,814,400 || trainable%: 0.021300285868169202


100%|██████████| 8/8 [00:00<00:00, 1818.27it/s]
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/jkarnuthala_umass_edu/.netrc.
wandb: Currently logged in as: jkarnuthala (jkarnuthala-university-of-massachusetts-amherst) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/home/jkarnuthala_umass_edu/.conda/envs/bo_intervention/lib/python3.11/site-packages/transformers/data/data_collator.py:656: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647429097/work/torch/csrc/utils/tensor_new.cpp:278.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)


Step,Training Loss
10,4.623300
20,1.677400
30,1.618300
40,1.374100
50,1.424900
60,1.447600
70,1.366200
80,1.298900
90,1.249000
100,1.217300


Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)



[QualEval] step 100
  target                generated
  laptop                'electronic'
  monkey                'monkey'
  electronic            'electronic'
  horse                 'horse'
  camel                 'dog'
  dog                   'cat'
  monitor               'cat'
  cat                   'dog'
  embed_sim (target vs generated): 0.6792 (mean over sample)



/home/jkarnuthala_umass_edu/.conda/envs/bo_intervention/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(



[QualEval] step 200
  target                generated
  laptop                'monitor'
  monkey                'horse'
  electronic            'camel'
  horse                 'horse'
  camel                 'horse'
  dog                   'monitor'
  monitor               'electronic'
  cat                   'horse'
  embed_sim (target vs generated): 0.4458 (mean over sample)


[QualEval] step 300
  target                generated
  laptop                'laptop'
  monkey                'cat'
  electronic            'laptop'
  horse                 'cat'
  camel                 'camel'
  dog                   'cat'
  monitor               'laptop'
  cat                   'laptop'
  embed_sim (target vs generated): 0.5907 (mean over sample)


[QualEval] step 400
  target                generated
  laptop                'horse'
  monkey                'dog'
  electronic            'horse'
  horse                 'dog'
  camel                 'camel'
  dog                   'horse'
  mo

### Quick multi-run 

Run this cell to try multiple values of a single hyperparameter back-to-back.
Each run is logged separately to W&B so you can compare them.

In [ ]:

sweep_param  = "layer"
sweep_values = [1,  4,  7, 10,  13, 16]

for val in sweep_values:
    run_name = f"{sweep_param}={val}"
    print(f"\n{'='*60}\nStarting run: {run_name}\n{'='*60}")
    train_semantle_reft(
        model_name        = MODEL_NAME,
        words             = words,
        intervention_type = INTERVENTION_TYPE,
        # word_sim_matrix   = word_sim_matrix,
        low_rank_dim      = LOW_RANK_DIM,
        layer             = val,
        position          = POSITION,
        epochs            = EPOCHS,
        batch_size        = BATCH_SIZE,
        lr                = LR,
        lr_scheduler_type = LR_SCHEDULER,
        warmup_ratio      = WARMUP_RATIO,
        beta              = BETA,
        lambda_sem        = LAMBDA_SEM,
        output_dir        = f"{OUTPUT_DIR}_sweep_{run_name}",
        cache_dir         = CACHE_DIR,
        wandb_project     = WANDB_PROJECT,
        wandb_run_name    = run_name,
        seed              = SEED,
        eval_steps        = EVAL_STEPS
    )
print("Sweep complete.")


Starting run: layer=1
trainable intervention params: 263,488 || trainable model params: 0
model params: 1,235,814,400 || trainable%: 0.0213210009528939


100%|██████████| 10/10 [00:00<00:00, 2232.68it/s]


Step,Training Loss
10,10.527500
20,10.280400
30,9.364700
40,9.178500
50,9.183400
60,8.806800
70,8.357100
80,7.840800
90,7.124200
100,6.589500



[QualEval] step 30
  target                generated
  computer              ''
  laptop                'is'
  monkey                ')'
  donkey                '"'
  tiger                 ')'
  horse                 ').'
  camel                 ''
  dog                   ').'


[QualEval] step 60
  target                generated
  computer              ''
  laptop                ''
  monkey                ':'
  donkey                ''
  tiger                 '<|end_of_text|>'
  horse                 'the'
  camel                 'S'
  dog                   '"'


[QualEval] step 90
  target                generated
  computer              '{"'
  laptop                'word'
  monkey                ''
  donkey                'a'
  tiger                 'the'
  horse                 'word'
  camel                 'the'
  dog                   '<|end_of_text|>'


[QualEval] step 120
  target                generated
  computer              ''
  laptop                '<|end_of_text|>'
 

100%|██████████| 10/10 [00:00<00:00, 2247.63it/s]


Step,Training Loss
10,10.106300
20,10.007600
30,8.926400
40,8.433100
50,8.174500
60,7.267300
70,6.399700
80,5.827500



[QualEval] step 30
  target                generated
  computer              '<|end_of_text|>'
  laptop                ''
  monkey                'You'
  donkey                ''
  tiger                 'You'
  horse                 '.'
  camel                 '<|end_of_text|>'
  dog                   'word'


[QualEval] step 60
  target                generated
  computer              ''
  laptop                ''
  monkey                'Your'
  donkey                ''
  tiger                 'word'
  horse                 'the'
  camel                 ''
  dog                   ''



KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7418ed494890>> (for post_run_cell), with arguments args (<ExecutionResult object at 7418e9fe0750, execution_count=21 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 7418e9fe2c10, raw_cell="# Sweep over beta values
sweep_param  = "layer"
sw.." transformed_cell="# Sweep over beta values
sweep_param  = "layer"
sw.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a226770753035312e756e6974792e72632e756d6173732e656475227d/work/pi_mccallum_umass_edu/jkarnuthala_umass_edu/D-Intervention/experiments.ipynb#X21sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost